In [1]:
%pip install pinecone-client sentence-transformers langchain pypdf langchain-text-splitters

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install langchain_community

Note: you may need to restart the kernel to use updated packages.


In [3]:
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader
 
loader = DirectoryLoader(
    "C:\\Users\\Prashant\\Desktop\\BIA\\Gen_AI\\Real_Time_Chatbot\\assets",        # path to folder
    glob="**/*.pdf",               # recursively loads PDFs
    loader_cls=PyPDFLoader         # use PyPDF loader for each file
)
 
docs = loader.load()
print("Total PDF pages:", len(docs))

c:\Users\Prashant\Desktop\BIA\Gen_AI\Real_Time_Chatbot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Total PDF pages: 16


In [4]:
docs

[Document(metadata={'producer': 'pdfTeX-1.40.18; modified using iText® 7.1.1 ©2000-2018 iText Group NV (AGPL-version)', 'creator': "'Certified by IEEE PDFeXpress at 06/22/2018 8:29:55 AM'", 'creationdate': '2018-06-22T15:27:43+00:00', 'ieee publication id': '8473398', 'meeting ending date': '17 Aug. 2018', 'trapped': 'False', 'moddate': '2018-10-11T10:24:48-04:00', 'subject': '2018 IEEE Conference on Computational Intelligence and Games (CIG);2018; ; ;', 'application': "'Certified by IEEE PDFeXpress at 06/22/2018 8:29:55 AM'", 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.18 (TeX Live 2017) kpathsea version 6.2.3', 'ieee issue id': '8490359', 'ieee article id': '8490433', 'title': 'Explainable AI for Designers: A Human-Centered Perspective on Mixed-Initiative Co-Creation', 'meeting starting date': '14 Aug. 2018', 'source': 'C:\\Users\\Prashant\\Desktop\\BIA\\Gen_AI\\Real_Time_Chatbot\\assets\\cig.2018.8490433.pdf', 'total_pages': 8, 'page': 0, 'page_label': '1'}, page

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
 
splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=100
)
 
chunks = splitter.split_documents(docs)
print("Total chunks:", len(chunks))

Total chunks: 303


In [6]:
chunks[100]

Document(metadata={'producer': 'pdfTeX-1.40.18; modified using iText® 7.1.1 ©2000-2018 iText Group NV (AGPL-version)', 'creator': "'Certified by IEEE PDFeXpress at 06/22/2018 8:29:55 AM'", 'creationdate': '2018-06-22T15:27:43+00:00', 'ieee publication id': '8473398', 'meeting ending date': '17 Aug. 2018', 'trapped': 'False', 'moddate': '2018-10-11T10:24:48-04:00', 'subject': '2018 IEEE Conference on Computational Intelligence and Games (CIG);2018; ; ;', 'application': "'Certified by IEEE PDFeXpress at 06/22/2018 8:29:55 AM'", 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.18 (TeX Live 2017) kpathsea version 6.2.3', 'ieee issue id': '8490359', 'ieee article id': '8490433', 'title': 'Explainable AI for Designers: A Human-Centered Perspective on Mixed-Initiative Co-Creation', 'meeting starting date': '14 Aug. 2018', 'source': 'C:\\Users\\Prashant\\Desktop\\BIA\\Gen_AI\\Real_Time_Chatbot\\assets\\cig.2018.8490433.pdf', 'total_pages': 8, 'page': 5, 'page_label': '6'}, page_

In [7]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')
 

In [8]:
model.save(r'C:\Users\Prashant\Desktop\BIA\Gen_AI\Real_Time_Chatbot\model\all-MiniLM-L6-v2')

In [9]:
vectors = [
    (str(i), model.encode(chunk.page_content).tolist(), {"text": chunk.page_content})
    for i, chunk in enumerate(chunks)
]

In [10]:
vectors

[('0',
  [-0.019360395148396492,
   0.026122737675905228,
   0.03255855664610863,
   -0.011248026043176651,
   0.01819206215441227,
   -0.039133500307798386,
   0.0185924731194973,
   -0.015518827363848686,
   0.020027916878461838,
   0.00285009341314435,
   -0.10305789113044739,
   -0.06369567662477493,
   0.0010535900946706533,
   -0.03884923830628395,
   -0.020300662145018578,
   -0.0026162208523601294,
   0.051625508815050125,
   -0.10186835378408432,
   -0.046710819005966187,
   -0.05111518129706383,
   0.0812881663441658,
   -0.03336106240749359,
   0.010087470524013042,
   -0.047535769641399384,
   -0.022643201053142548,
   0.08925504237413406,
   0.06610953062772751,
   -0.037912704050540924,
   0.07101196050643921,
   -0.041289299726486206,
   0.03505932167172432,
   0.049750812351703644,
   0.038126200437545776,
   0.0174107626080513,
   -0.032814882695674896,
   0.06989023834466934,
   -0.028240296989679337,
   0.03808530792593956,
   0.07883648574352264,
   -0.0340430252254

In [11]:
%pip install pinecone

Note: you may need to restart the kernel to use updated packages.


In [12]:
import os
from dotenv import load_dotenv
from pinecone import Pinecone
 
load_dotenv()
 
pc = Pinecone(api_key="pcsk_5KfdhQ_EZ83VeRLvTmTSffhTVjQExLx3E2HtKDdsNytX6z2JFsvqzjeRi81wAcYCvpJJd3")
index = pc.Index("data-ingesstion-v2")

In [13]:
for chunk in chunks:
    index.upsert(
        vectors=vectors,
    )

In [14]:
query = "What does the document say about risk factors?"
qvec = model.encode(query).tolist()
 
res = index.query(
    vector=qvec,
    top_k=5,
    include_metadata=True
)
 
for r in res.matches:
    print(r.metadata['text'], "\n---")

possible alternatives, sampling methods used, evaluation
criteria);
• sketch a range of choices (e.g. characterize the extreme
points of a range of options, to give insight into what it
involves);
• warn a designer regarding some risk ahead (e.g. look
ahead for what-if analysis, to identify and describe con-
ﬂicts or risks)
The explanations mentioned in these examples require a con- 
---
ﬂicts or risks)
The explanations mentioned in these examples require a con-
siderable understanding of the processes and goals at hand.
Going even higher on the spectrum of autonomy and
initiative, we can devise an AI system working on par with
the human designer, taking on activities more as a colleague
than as an AI assistant. At this level the tasks, outcomes and 
---
affect future outcomes, or to foreshadow how one early
decision affects the ﬁnal outcome.
• summary of highlights of the generative process, by
ﬁltering out and omitting less interesting points in the
generated sentence structure. For 

In [15]:
%pip install flask groq

Note: you may need to restart the kernel to use updated packages.
